# Evolutionäres Constrain basiertes NAS

- Dataset: EuroSAT
- Model: timm ViT-Small pretrained (ImageNet)
- Search space: depth, heads, patch_size (prepared), SPT/LSA flags (prepared),
                finetune strategy + optimizer hyperparams
- NAS: population -> elite selection -> mutation -> generations
- Constraints: soft penalty (val_acc - lambda*params - mu*time)

### Imports

In [ ]:
import time
import csv
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

/Users/matsadel/Desktop/Master Informatik/Semester 3/Hauptprojekt/hpenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)

Using device: mps


In [3]:
def make_eurosat_loaders(
    root="./data",
    image_size=224,
    batch_size=16,
    num_workers=2,
    seed=42,
):
    tfm = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        ),
    ])
    
    ds = datasets.EuroSAT(root=root, download=True, transform=tfm)
    
    n = len(ds)
    n_train = int(0.7 * n)
    n_val = int(0.15 * n)
    n_test = n - n_train - n_val
    
    g = torch.Generator().manual_seed(seed)
    train_ds, val_ds, test_ds = random_split(
        ds, [n_train, n_val, n_test], generator=g
    )
    
    use_pin = torch.cuda.is_available()
    
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=use_pin
    )
    
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=use_pin
    )
    
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=use_pin
    )
    
    return train_loader, val_loader, test_loader, len(ds.classes)

train_loader, val_loader, test_loader, num_classes = make_eurosat_loaders()
print("Classes:", num_classes)

Classes: 10


In [ ]:
class SPTPatchEmbed(nn.Module):
    """
    Shifted Patch Tokenization (SPT) as Patch Embedding replacement for ViT.
    Produces patch tokens with increased locality inductive bias by concatenating
    shifted versions of the input image along channel dimension.
    """
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_chans=3,
        embed_dim=384,
        vanilla=False,
        use_ln=True,
        eps=1e-6
    ):
        super().__init__()
        self.img_size = img_size if isinstance(img_size, int) else img_size[0]
        self.patch_size = patch_size if isinstance(patch_size, int) else patch_size[0]
        self.half_patch = self.patch_size // 2
        self.vanilla = vanilla
        self.use_ln = use_ln and (not vanilla)
        
        effective_in_chans = in_chans if vanilla else in_chans * 5
        
        self.proj = nn.Conv2d(
            effective_in_chans,
            embed_dim,
            kernel_size=self.patch_size,
            stride=self.patch_size
        )
        
        self.norm = nn.LayerNorm(embed_dim, eps=eps) if self.use_ln else nn.Identity()
        
    def _crop_shift_pad(self, x, mode: str):
        hp = self.half_patch
        B, C, H, W = x.shape
        
        if mode == "left-up":
            top_crop, left_crop = hp, hp
            top_pad, left_pad = 0, 0
        elif mode == "left-down":
            top_crop, left_crop = 0, hp
            top_pad, left_pad = hp, 0
        elif mode == "right-up":
            top_crop, left_crop = hp, 0
            top_pad, left_pad = 0, hp
        elif mode == "right-down":
            top_crop, left_crop = 0, 0
            top_pad, left_pad = hp, hp
        else:
            raise ValueError(mode)
        
        x_crop = x[:, :, top_crop + (H - hp), left_crop:left_crop + (W - hp)]
        x_pad = F.pad(x_crop, (left_pad, W - (W - hp) - left_pad, top_pad, H - (H - hp) - top_pad))
        
        return x_pad
    
    def forward(self, x):
        if not self.vanilla:
            x = torch.cat(
                [
                x, 
                self._crop_shift_pad(x, "left-up"),
                self._crop_shift_pad(x, "left-down"),
                self._crop_shift_pad(x, "right-up"),
                self._crop_shift_pad(x, "right-down"),
                ],
                dim = 1
            )
        
        x = self.proj(x)
        
        x = x.flatten(2).transpose(1, 2)
        
        x = self.norm(x)
        
        return x
        
    

In [ ]:
def enable_spt(model, img_size=224, patch_size=16, vanilla=False):
    """
    Replace model.patch_embed with SPTPatchEmbed while keeping everything else identical.
    Assumes ViT-Small (embed_dim=384).
    """
    embed_dim = getattr(model, "embed_dim", 384)
    in_chans = 3
    
    model.patch_embed = SPTPatchEmbed(
        img_size=img_size,
        patch_size=patch_size,
        in_chans=in_chans,
        embed_dim=embed_dim,
        vanilla=vanilla,
        use_ln=True
    )
    
    return model

In [ ]:
def build_vit_from_cfg(num_classes, cfg):
    """
    Builds a ViT-Small using timm with variable depth and num_heads (if supported by your timm version).
    Notes:
    - patch_size is prepared in cfg but not applied yet due to pretrained compatibility complexity.
    - use_spt/use_lsa are flags prepared for later module integration.
    """
    model = timm.create_model(
        "vit_small_patch16_224",
        pretrained=True,
        num_classes=num_classes,
        depth=cfg["depth"],
        num_heads=cfg["num_heads"],
    )

    if cfg["use_spt"] == 1:
        model = enable_spt(model, img_size=224, patch_size=16, vanilla=False)
        
    # TODO later:
    # if cfg["use_lsa"] == 1: model = replace_attention_with_lsa(model)

    mode = cfg.get("finetune_mode", "last2")

    # Freeze all
    for p in model.parameters():
        p.requires_grad = False

    # Train head always
    for p in model.head.parameters():
        p.requires_grad = True

    # Finetune modes
    if mode == "full":
        for p in model.parameters():
            p.requires_grad = True
    elif mode == "last2":
        for blk in model.blocks[-2:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in model.norm.parameters():
            p.requires_grad = True
    elif mode == "head":
        pass
    else:
        raise ValueError(mode)

    return model.to(device)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

In [5]:
def accuracy_top1(logits, y):
    return (logits.argmax(1) == y).float().mean().item()

def train_one_epoch(model, loader, optimizer):
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    total_loss, total_acc, n = 0.0, 0.0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_acc += accuracy_top1(logits, y) * bs
        n += bs

    return total_loss / n, total_acc / n

@torch.no_grad()
def eval_one_epoch(model, loader):
    model.eval()
    loss_fn = nn.CrossEntropyLoss()
    total_loss, total_acc, n = 0.0, 0.0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = loss_fn(logits, y)

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_acc += accuracy_top1(logits, y) * bs
        n += bs

    return total_loss / n, total_acc / n

def quick_finetune(model, train_loader, val_loader, epochs=2, lr=3e-4, weight_decay=1e-4):
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)

    start = time.time()
    best_val_acc = 0.0

    for _ in range(epochs):
        train_one_epoch(model, train_loader, optimizer)
        _, val_acc = eval_one_epoch(model, val_loader)
        best_val_acc = max(best_val_acc, val_acc)

    return best_val_acc, time.time() - start

In [ ]:
SEARCH_SPACE = {
    "patch_size": [16],            # later: [16, 8] with proper pretrained adaptation
    "depth": [6, 8, 10, 12],
    "num_heads": [4, 6, 8],        # must divide embed_dim=384
    "use_spt": [0, 1],             # prepared (not yet applied)
    "use_lsa": [0, 1],             # prepared (not yet applied)
    "finetune_mode": ["head", "last2", "full"],
    "lr": [1e-4, 3e-4, 1e-3],
    "weight_decay": [1e-4, 5e-4],
    "e": [2],
}

def sample_config(space=SEARCH_SPACE):
    cfg = {k: random.choice(v) for k, v in space.items()}

    # ensure heads divides embed_dim
    if 384 % cfg["num_heads"] != 0:
        valid = [h for h in space["num_heads"] if 384 % h == 0]
        cfg["num_heads"] = random.choice(valid)

    return cfg

def mutate_config(cfg, space=SEARCH_SPACE, p_mut=0.35):
    child = cfg.copy()
    for key, choices in space.items():
        if random.random() < p_mut:
            options = [c for c in choices if c != child[key]]
            child[key] = random.choice(options) if options else child[key]

    # ensure validity again
    if 384 % child["num_heads"] != 0:
        valid = [h for h in space["num_heads"] if 384 % h == 0]
        child["num_heads"] = random.choice(valid)

    return child

In [7]:
def fitness(val_acc, params, time_sec, lam_params=1e-8, lam_time=0.02):
    # val_acc in 0..1
    # params in absolute count
    # time_sec -> penalty per minute
    return val_acc - lam_params * params - lam_time * (time_sec / 60.0)

def evaluate_cfg(cfg):
    model = build_vit_from_cfg(num_classes, cfg)

    params = count_params(model)
    val_acc, secs = quick_finetune(
        model, train_loader, val_loader,
        epochs=cfg["proxy_epochs"],
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"]
    )

    fit = fitness(val_acc, params, secs)

    return {
        **cfg,
        "val_acc": val_acc,
        "time_sec": secs,
        "params": params,
        "fitness": fit
    }

In [8]:
def evolutionary_nas(
    pop_size=8,
    generations=5,
    elite_k=3,
    p_mut=0.35,
    seed=42,
    log_csv_path="enas_eurosat.csv"
):
    random.seed(seed)

    population = [sample_config() for _ in range(pop_size)]
    all_results = []

    fieldnames = list(SEARCH_SPACE.keys()) + ["val_acc", "time_sec", "params", "fitness", "generation", "rank"]

    with open(log_csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for gen in range(generations):
            print(f"\n=== Generation {gen} ===")
            gen_results = []

            for i, cfg in enumerate(population):
                res = evaluate_cfg(cfg)
                gen_results.append(res)
                print(
                    f"  [{i:02d}] "
                    f"val_acc={res['val_acc']:.4f} "
                    f"fitness={res['fitness']:.4f} "
                    f"params={res['params']/1e6:.1f}M "
                    f"time={res['time_sec']:.1f}s "
                    f"cfg(depth={res['depth']}, heads={res['num_heads']}, spt={res['use_spt']}, lsa={res['use_lsa']}, ft={res['finetune_mode']})"
                )

            gen_results = sorted(gen_results, key=lambda x: x["fitness"], reverse=True)

            for rank, res in enumerate(gen_results):
                row = res.copy()
                row["generation"] = gen
                row["rank"] = rank
                writer.writerow(row)

            all_results.extend([{**r, "generation": gen} for r in gen_results])

            elites = gen_results[:elite_k]

            # next population: mutate elites
            new_population = []
            while len(new_population) < pop_size:
                parent = random.choice(elites)
                parent_cfg = {k: parent[k] for k in SEARCH_SPACE.keys()}
                child_cfg = mutate_config(parent_cfg, p_mut=p_mut)
                new_population.append(child_cfg)

            population = new_population

    return all_results

In [9]:
results = evolutionary_nas(
    pop_size=8,
    generations=5,
    elite_k=3,
    p_mut=0.35,
    log_csv_path="enas_eurosat.csv"
)

print("\nDone. Results written to enas_eurosat.csv")


=== Generation 0 ===


Unexpected keys (blocks.10.attn.proj.bias, blocks.10.attn.proj.weight, blocks.10.attn.qkv.bias, blocks.10.attn.qkv.weight, blocks.10.mlp.fc1.bias, blocks.10.mlp.fc1.weight, blocks.10.mlp.fc2.bias, blocks.10.mlp.fc2.weight, blocks.10.norm1.bias, blocks.10.norm1.weight, blocks.10.norm2.bias, blocks.10.norm2.weight, blocks.11.attn.proj.bias, blocks.11.attn.proj.weight, blocks.11.attn.qkv.bias, blocks.11.attn.qkv.weight, blocks.11.mlp.fc1.bias, blocks.11.mlp.fc1.weight, blocks.11.mlp.fc2.bias, blocks.11.mlp.fc2.weight, blocks.11.norm1.bias, blocks.11.norm1.weight, blocks.11.norm2.bias, blocks.11.norm2.weight, blocks.6.attn.proj.bias, blocks.6.attn.proj.weight, blocks.6.attn.qkv.bias, blocks.6.attn.qkv.weight, blocks.6.mlp.fc1.bias, blocks.6.mlp.fc1.weight, blocks.6.mlp.fc2.bias, blocks.6.mlp.fc2.weight, blocks.6.norm1.bias, blocks.6.norm1.weight, blocks.6.norm2.bias, blocks.6.norm2.weight, blocks.7.attn.proj.bias, blocks.7.attn.proj.weight, blocks.7.attn.qkv.bias, blocks.7.attn.qkv.weight,

  [00] val_acc=0.8701 fitness=0.7016 params=11.0M time=175.0s cfg(depth=6, heads=8, spt=1, lsa=0, ft=head)


Unexpected keys (blocks.10.attn.proj.bias, blocks.10.attn.proj.weight, blocks.10.attn.qkv.bias, blocks.10.attn.qkv.weight, blocks.10.mlp.fc1.bias, blocks.10.mlp.fc1.weight, blocks.10.mlp.fc2.bias, blocks.10.mlp.fc2.weight, blocks.10.norm1.bias, blocks.10.norm1.weight, blocks.10.norm2.bias, blocks.10.norm2.weight, blocks.11.attn.proj.bias, blocks.11.attn.proj.weight, blocks.11.attn.qkv.bias, blocks.11.attn.qkv.weight, blocks.11.mlp.fc1.bias, blocks.11.mlp.fc1.weight, blocks.11.mlp.fc2.bias, blocks.11.mlp.fc2.weight, blocks.11.norm1.bias, blocks.11.norm1.weight, blocks.11.norm2.bias, blocks.11.norm2.weight, blocks.6.attn.proj.bias, blocks.6.attn.proj.weight, blocks.6.attn.qkv.bias, blocks.6.attn.qkv.weight, blocks.6.mlp.fc1.bias, blocks.6.mlp.fc1.weight, blocks.6.mlp.fc2.bias, blocks.6.mlp.fc2.weight, blocks.6.norm1.bias, blocks.6.norm1.weight, blocks.6.norm2.bias, blocks.6.norm2.weight, blocks.7.attn.proj.bias, blocks.7.attn.proj.weight, blocks.7.attn.qkv.bias, blocks.7.attn.qkv.weight,

  [01] val_acc=0.8936 fitness=0.7306 params=11.0M time=158.2s cfg(depth=6, heads=4, spt=0, lsa=0, ft=head)


...

  [05] val_acc=0.9257 fitness=0.6730 params=18.1M time=214.6s cfg(depth=10, heads=6, spt=0, lsa=1, ft=head)


Unexpected keys (blocks.10.attn.proj.bias, blocks.10.attn.proj.weight, blocks.10.attn.qkv.bias, blocks.10.attn.qkv.weight, blocks.10.mlp.fc1.bias, blocks.10.mlp.fc1.weight, blocks.10.mlp.fc2.bias, blocks.10.mlp.fc2.weight, blocks.10.norm1.bias, blocks.10.norm1.weight, blocks.10.norm2.bias, blocks.10.norm2.weight, blocks.11.attn.proj.bias, blocks.11.attn.proj.weight, blocks.11.attn.qkv.bias, blocks.11.attn.qkv.weight, blocks.11.mlp.fc1.bias, blocks.11.mlp.fc1.weight, blocks.11.mlp.fc2.bias, blocks.11.mlp.fc2.weight, blocks.11.norm1.bias, blocks.11.norm1.weight, blocks.11.norm2.bias, blocks.11.norm2.weight) found while loading pretrained weights. This may be expected if model is being adapted.


  [06] val_acc=0.9704 fitness=0.6995 params=18.1M time=268.9s cfg(depth=10, heads=6, spt=1, lsa=1, ft=last2)


Unexpected keys (blocks.10.attn.proj.bias, blocks.10.attn.proj.weight, blocks.10.attn.qkv.bias, blocks.10.attn.qkv.weight, blocks.10.mlp.fc1.bias, blocks.10.mlp.fc1.weight, blocks.10.mlp.fc2.bias, blocks.10.mlp.fc2.weight, blocks.10.norm1.bias, blocks.10.norm1.weight, blocks.10.norm2.bias, blocks.10.norm2.weight, blocks.11.attn.proj.bias, blocks.11.attn.proj.weight, blocks.11.attn.qkv.bias, blocks.11.attn.qkv.weight, blocks.11.mlp.fc1.bias, blocks.11.mlp.fc1.weight, blocks.11.mlp.fc2.bias, blocks.11.mlp.fc2.weight, blocks.11.norm1.bias, blocks.11.norm1.weight, blocks.11.norm2.bias, blocks.11.norm2.weight, blocks.8.attn.proj.bias, blocks.8.attn.proj.weight, blocks.8.attn.qkv.bias, blocks.8.attn.qkv.weight, blocks.8.mlp.fc1.bias, blocks.8.mlp.fc1.weight, blocks.8.mlp.fc2.bias, blocks.8.mlp.fc2.weight, blocks.8.norm1.bias, blocks.8.norm1.weight, blocks.8.norm2.bias, blocks.8.norm2.weight, blocks.9.attn.proj.bias, blocks.9.attn.proj.weight, blocks.9.attn.qkv.bias, blocks.9.attn.qkv.weight,

KeyboardInterrupt: 